[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/06_multihead_attention.ipynb)

# 🔴 Hard: Multi-Head Attention

Implement **Multi-Head Attention** from scratch — the core building block of the Transformer.

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W^O$$
$$\text{head}_i = \text{Attention}(Q W_i^Q,\; K W_i^K,\; V W_i^V)$$

### Signature
```python
class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, Q, K, V) -> torch.Tensor: ...
```

### Requirements
- Use `nn.Linear(d_model, d_model)` for `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`
- `d_k = d_model // num_heads` per head
- `forward(Q, K, V)`: Q is `(B, seq_q, d_model)`, K/V are `(B, seq_k, d_model)`
- Must support **cross-attention** (`seq_q != seq_k`)
- Do **NOT** use `torch.nn.MultiheadAttention`
- You **may** use `torch.softmax` and `torch.matmul`

### Steps
1. Project: `q = self.W_q(Q)`, `k = self.W_k(K)`, `v = self.W_v(V)`
2. Reshape to `(B, num_heads, seq, d_k)`
3. Scaled dot-product attention per head
4. Concat heads → `(B, seq_q, d_model)`
5. Output projection: `self.W_o(concat)`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.7 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [27]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadAttention:
    def __init__(self, d_model: int, num_heads: int):
      self.W_q = nn.Linear(d_model, d_model, bias=False)
      self.W_k = nn.Linear(d_model, d_model, bias=False)
      self.W_v = nn.Linear(d_model, d_model, bias=False)
      self.W_o = nn.Linear(d_model, d_model, bias=False)
      self.d_model = d_model
      assert(d_model % num_heads == 0, "d_model must be devisible by num_heads")
      self.d_head = d_model // num_heads
      self.num_heads = num_heads

    def forward(self, Q, K, V): # Q, K, V = [B, Sq|Sk, D]
      B, Sq, D = Q.shape
      Bk, Sk, Dk = K.shape
      Bv, Sv, Dv = V.shape
      assert(B == Bk == Bv, "Batch size must be all equal")
      assert(Sk == Sv, "Sequence for K and V shall be the same")
      assert(D == Dk == Dv == self.d_model, "QKV shall have the same dimensions")

      q = self.W_q(Q)
      k = self.W_k(K)
      v = self.W_v(V)

      q = q.view(B, Sq, self.num_heads, self.d_head).transpose(1, 2) # [B, H, Sq, Dh]
      k = k.view(B, Sk, self.num_heads, self.d_head).transpose(1, 2)
      v = v.view(B, Sk, self.num_heads, self.d_head).transpose(1, 2)

      # Note here is the d_head not num_head...
      attn = torch.softmax(q @ k.mT / math.sqrt(self.d_head), dim=-1) @ v # [B, H, Sq, Dh]
      attn = attn.transpose(1, 2).contiguous().view(B, Sq, self.d_model) #[B, H, Sq, Dh] -> [B, Sq, H, Dh] -> [B, Sq, D]
      return self.W_o(attn)


<>:10: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:18: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:19: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:20: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:10: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:18: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:19: SyntaxWarning: assertion is always true, perhaps remove parentheses?
<>:20: SyntaxWarning: assertion is always true, perhaps remove parentheses?
/tmp/ipykernel_5351/4148198030.py:10: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(d_model % num_heads == 0, "d_model must be devisible by num_heads")
/tmp/ipykernel_5351/4148198030.py:18: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(B == Bk == Bv, "Batch size must be all equal")
/tmp/ipykernel_5351/4148198030.py:19: SyntaxW

In [23]:
# 🧪 Debug
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, num_heads=4)
print("W_q type:", type(mha.W_q))          # should be nn.Linear
print("W_q.weight shape:", mha.W_q.weight.shape)  # (32, 32)

x = torch.randn(2, 6, 32)
out = mha.forward(x, x, x)
print("Output shape:", out.shape)          # (2, 6, 32)

# Cross-attention
Q = torch.randn(1, 3, 32)
K = torch.randn(1, 7, 32)
V = torch.randn(1, 7, 32)
out2 = mha.forward(Q, K, V)
print("Cross-attn shape:", out2.shape)     # (1, 3, 32)

B = 2
Sq = 4
Sk = 5
D = 8
H = 2

mha = MultiHeadAttention(D, H)

Q = torch.randn(B, Sq, D)
K = torch.randn(B, Sk, D)
V = torch.randn(B, Sk, D)

out = mha.forward(Q, K, V)

assert out.shape == (B, Sq, D)

x = torch.randn(B, Sq, D)

out = mha.forward(x, x, x)

assert out.shape == (B, Sq, D)
assert torch.isfinite(out).all()

B, Sq, Sk, D = 2, 3, 4, 8

mha = MultiHeadAttention(D, num_heads=1)

Q = torch.randn(B, Sq, D)
K = torch.randn(B, Sk, D)
V = torch.randn(B, Sk, D)

out = mha.forward(Q, K, V)

assert out.shape == (B, Sq, D)

B, D, H = 2, 8, 2

mha = MultiHeadAttention(D, H)

Q = torch.randn(B, 1, D)
K = torch.randn(B, 1, D)
V = torch.randn(B, 1, D)

out = mha.forward(Q, K, V)

assert out.shape == (B, 1, D)

B = 1
Sq = 3
Sk = 4
D = 4
H = 2

mha = MultiHeadAttention(D, H)

with torch.no_grad():
    mha.W_q.weight.zero_()
    mha.W_k.weight.zero_()

    mha.W_v.weight.copy_(torch.eye(D))
    mha.W_o.weight.copy_(torch.eye(D))

Q = torch.randn(B, Sq, D)
K = torch.randn(B, Sk, D)

V = torch.tensor([
    [
        [1., 2., 3., 4.],
        [2., 3., 4., 5.],
        [3., 4., 5., 6.],
        [4., 5., 6., 7.],
    ]
])

out = mha.forward(Q, K, V)

expected_one_token = V.mean(dim=1, keepdim=True)
expected = expected_one_token.expand(B, Sq, D)

torch.testing.assert_close(out, expected)

B = 2
Sq = 3
Sk = 5
D = 8
H = 2

mha = MultiHeadAttention(D, H)

with torch.no_grad():
    mha.W_v.weight.copy_(torch.eye(D))
    mha.W_o.weight.copy_(torch.eye(D))

Q = torch.randn(B, Sq, D)
K = torch.randn(B, Sk, D)

v = torch.randn(B, 1, D)
V = v.expand(B, Sk, D)

out = mha.forward(Q, K, V)

expected = v.expand(B, Sq, D)

torch.testing.assert_close(
    out,
    expected,
    atol=1e-5,
    rtol=1e-5,
)

W_q type: <class 'torch.nn.modules.linear.Linear'>
W_q.weight shape: torch.Size([32, 32])
Output shape: torch.Size([2, 6, 32])
Cross-attn shape: torch.Size([1, 3, 32])


In [28]:
# ✅ SUBMIT
from torch_judge import check
check("mha")


🧪 Testing: Multi-Head Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/6] Output shape (5.7ms)
  ✅ [2/6] Uses nn.Linear with correct shapes (0.7ms)
  ✅ [3/6] Numerical correctness vs reference (3.3ms)
  ✅ [4/6] Gradient flow (2.7ms)
  ✅ [5/6] Cross-attention (seq_q != seq_k) (0.7ms)
  ✅ [6/6] Different heads give different outputs (2.0ms)
──────────────────────────────────────────────────
  🎉 All 6 tests passed! (15.2ms total)
  Progress saved. Run status() to see your dashboard.

